<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Fraude, Consumo de Agua y Ajuste sin Trampa 🎯
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 03
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/03_Metricas_y_Ajuste_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 03: Métricas y Ajuste de Hiperparámetros** ([README del módulo](../03%20-%20Metricas%20y%20Ajuste%20de%20Hiperparametros/README.md)) con dos problemas **nuevos**: detectar **fraude** en transacciones con tarjeta (clasificación muy desbalanceada) y predecir el **consumo de agua** de un sector (regresión con valores atípicos).

Al terminar serás capaz de:

1. **Construir a mano** la matriz de confusión y las métricas de clasificación, y detectar la **paradoja de la exactitud**.
2. **Elegir el umbral de decisión** minimizando un costo asimétrico, sin tocar el conjunto de prueba, y distinguir ROC-AUC de PR-AUC.
3. **Calcular a mano** MSE, RMSE, MAE, MAPE y $R^2$ y compararlas contra una línea base ingenua.
4. **Ajustar hiperparámetros** con `GridSearchCV` dentro de un `Pipeline` y validación cruzada estratificada, y reportar la brecha entre validación y prueba.

> 📚 **Repasa si lo necesitas:** [Métricas de Clasificación](../03%20-%20Metricas%20y%20Ajuste%20de%20Hiperparametros/00_Metricas_de_Clasificacion.ipynb) · [Métricas de Regresión](../03%20-%20Metricas%20y%20Ajuste%20de%20Hiperparametros/01_Metricas_de_Regresion.ipynb) · [Validación Cruzada y GridSearch](../03%20-%20Metricas%20y%20Ajuste%20de%20Hiperparametros/02_Validacion_Cruzada_y_GridSearch.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija). Bibliotecas: `numpy`, `pandas`, `scikit-learn`. En los retos 1 y 3 calcularás las métricas **con NumPy**; `sklearn.metrics` queda para las autoverificaciones y los demás retos.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, mean_squared_error, mean_absolute_error,
                             mean_absolute_percentage_error, r2_score)

warnings.filterwarnings("ignore")


def generar_transacciones(n=6000, seed=33):
    rng = np.random.default_rng(seed)
    monto = rng.lognormal(10.5, 1.0, n)                 # COP
    hora = rng.integers(0, 24, n).astype(float)
    dist = rng.gamma(1.5, 8, n)                         # km entre el comercio y el domicilio
    n24 = rng.poisson(2.0, n).astype(float)             # transacciones en las últimas 24 h
    antig = rng.uniform(0, 120, n)                      # meses como cliente
    online = rng.binomial(1, 0.4, n).astype(float)
    ruido = rng.normal(0, 1, n)
    z = (-5.6 + 1.3 * (np.log(monto) - 10.5) + 0.05 * dist + 0.45 * n24 - 0.02 * antig
         + 1.0 * online + 1.6 * ((hora < 6) | (hora >= 23)))
    y = rng.binomial(1, 1 / (1 + np.exp(-z)))
    return pd.DataFrame({"monto_cop": monto, "hora": hora, "distancia_km": dist, "tx_24h": n24,
                         "antiguedad_meses": antig, "es_online": online, "ruido": ruido, "fraude": y})


def generar_agua(n=1200, seed=5):
    rng = np.random.default_rng(seed)
    temp = rng.normal(18, 5, n)
    lluvia = rng.exponential(3, n)
    viviendas = rng.integers(80, 600, n).astype(float)
    festivo = rng.binomial(1, 0.12, n).astype(float)
    y = 40 + 0.45 * viviendas * (1 + 0.03 * (temp - 18)) - 1.5 * lluvia + 35 * festivo + rng.normal(0, 18, n)
    y = y + (rng.random(n) < 0.02) * rng.normal(150, 30, n)       # 2 % de lecturas con fugas / errores
    return pd.DataFrame({"temperatura_c": temp, "lluvia_mm": lluvia, "viviendas": viviendas,
                         "festivo": festivo, "consumo_m3": y})


# Clasificación: fraude (≈ 5 % de positivos)
df = generar_transacciones()
X, y = df.drop(columns="fraude"), df["fraude"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# Regresión: consumo de agua (m³ por día)
dg = generar_agua()
Xr_train, Xr_test, yr_train, yr_test = train_test_split(dg.drop(columns="consumo_m3"), dg["consumo_m3"], test_size=0.25, random_state=0)

print("Fraude: entrenamiento", X_train.shape, "| prueba", X_test.shape, "| proporción de fraudes:", round(y.mean(), 3))
print("Agua: entrenamiento", Xr_train.shape, "| prueba", Xr_test.shape)

---
## Reto 1: La matriz de confusión a mano y la paradoja de la exactitud 🧩

**Contexto:** el banco quiere un detector de fraude. Un directivo ve «95 % de exactitud» y celebra. Tú vas a comprobar qué hay detrás.

**Enunciado**
1. Ajusta `modelo = Pipeline([StandardScaler, LogisticRegression(max_iter=1000)])` con el entrenamiento y obtén `y_pred = modelo.predict(X_test)` (umbral 0.5).
2. Con **NumPy** (comparaciones booleanas y `np.sum`, sin `sklearn.metrics`) calcula los conteos enteros `tp`, `fp`, `fn`, `tn` (la clase positiva es 1 = fraude).
3. Calcula con las fórmulas: `exactitud`, `precision`, `recall` y `f1` (usa $F_1 = \dfrac{2\,P\,R}{P + R}$).
4. `acc_trivial`: exactitud de un «modelo» que predice **siempre 0**.

**Criterios de aceptación:** conteos y métricas idénticos a los de `scikit-learn`; `acc_trivial` es la proporción de la clase mayoritaria; y la exactitud de tu modelo supera a `acc_trivial` por menos de 0.03 (la paradoja: casi no mejora a no hacer nada).

**Puntaje sugerido:** 20 puntos.

In [ ]:
# TODO: completa
modelo = None    # Pipeline(StandardScaler, LogisticRegression(max_iter=1000)) ajustado
y_pred = None    # predicciones de clase en prueba

tp = fp = fn = tn = None                     # enteros, calculados con NumPy
exactitud = precision = recall = f1 = None   # fórmulas a partir de tp, fp, fn, tn
acc_trivial = None                           # exactitud de predecir siempre 0

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert isinstance(modelo, Pipeline) and hasattr(modelo[-1], "coef_"), "'modelo' debe ser un Pipeline ajustado que termina en LogisticRegression."
assert y_pred is not None and np.array_equal(np.asarray(y_pred), modelo.predict(X_test)), "y_pred debe ser modelo.predict(X_test)."
_tn, _fp, _fn, _tp = confusion_matrix(y_test, y_pred).ravel()
assert (tn, fp, fn, tp) == (_tn, _fp, _fn, _tp), "Los conteos no coinciden con la matriz de confusión: tp = real 1 y predicho 1; fp = real 0 y predicho 1; fn = real 1 y predicho 0; tn = real 0 y predicho 0."
assert abs(exactitud - accuracy_score(y_test, y_pred)) < 1e-9, "exactitud = (tp + tn) / total."
assert abs(precision - precision_score(y_test, y_pred, zero_division=0)) < 1e-9, "precision = tp / (tp + fp)."
assert abs(recall - recall_score(y_test, y_pred)) < 1e-9, "recall = tp / (tp + fn)."
assert abs(f1 - f1_score(y_test, y_pred, zero_division=0)) < 1e-9, "f1 = 2PR / (P + R)."
assert acc_trivial is not None and abs(acc_trivial - (1 - y_test.mean())) < 1e-9, "acc_trivial es la exactitud de predecir siempre 0: la proporción de ceros en y_test."
assert 0 <= exactitud - acc_trivial < 0.03, "Con este desbalance la exactitud del modelo debería superar a acc_trivial por menos de 0.03 (la paradoja de la exactitud)."
print(f"✅ Reto 1 superado: exactitud = {exactitud:.3f} (trivial: {acc_trivial:.3f}) pero recall = {recall:.3f} y F1 = {f1:.3f}")

---
## Reto 2: Elegir el umbral por costo, sin mirar la prueba 💰

**Contexto:** dejar pasar un fraude cuesta en promedio **\$200** (falso negativo); bloquear una compra legítima cuesta **\$5** de gestión (falso positivo). Con costos tan asimétricos, el umbral 0.5 no es el adecuado. Tampoco vale elegirlo mirando la prueba: lo harás con probabilidades de **validación cruzada** del entrenamiento.

**Enunciado**
1. `auc` (ROC-AUC) y `ap` (PR-AUC, `average_precision_score`) del `modelo` en el conjunto de prueba.
2. Implementa `costo(y_real, y_pred, c_fn=200, c_fp=5)`: costo total $= c_{fn} \cdot FN + c_{fp} \cdot FP$.
3. `proba_cv`: probabilidades de la clase 1 de `cross_val_predict(modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1]`.
4. Recorre `umbrales = np.linspace(0.01, 0.99, 99)`, calcula `costos_cv` (costo en entrenamiento al predecir 1 si `proba_cv >= umbral`) y guarda en `umbral_opt` el umbral de menor costo.
5. `costo_05` y `costo_opt`: costo en **prueba** con el umbral 0.5 y con `umbral_opt`.

**Criterios de aceptación:** `costo` correcta; `umbral_opt` es mucho menor que 0.5; el costo en prueba baja con `umbral_opt`; y la PR-AUC es menor que la ROC-AUC (con clases raras la PR-AUC es más exigente).

**Puntaje sugerido:** 20 puntos.

In [ ]:
def costo(y_real, y_pred, c_fn=200, c_fp=5):
    # TODO: costo total = c_fn * (# falsos negativos) + c_fp * (# falsos positivos)
    return None


# TODO: completa
auc = None          # ROC-AUC de `modelo` en prueba
ap = None           # PR-AUC (average_precision_score) de `modelo` en prueba

proba_cv = None     # probabilidades de clase 1 vía cross_val_predict sobre el entrenamiento
umbrales = np.linspace(0.01, 0.99, 99)
costos_cv = None    # lista/arreglo con el costo en entrenamiento para cada umbral
umbral_opt = None   # umbral de menor costo

costo_05 = None     # costo en prueba con umbral 0.5
costo_opt = None    # costo en prueba con umbral_opt

In [ ]:
# Autoverificación del Reto 2 (no modificar)
_c = costo(np.array([1, 0, 1, 0]), np.array([0, 1, 1, 0]))
assert _c is not None and _c == 205, "costo([1,0,1,0], [0,1,1,0]) debe ser 205 (1 FN × 200 + 1 FP × 5)."
_p = modelo.predict_proba(X_test)[:, 1]
assert auc is not None and abs(auc - roc_auc_score(y_test, _p)) < 1e-9, "auc debe ser el ROC-AUC de 'modelo' en prueba (con probabilidades, no con clases)."
assert ap is not None and abs(ap - average_precision_score(y_test, _p)) < 1e-9, "ap debe ser average_precision_score(y_test, probabilidades)."
assert ap < auc, "Con clases raras la PR-AUC debería ser menor que la ROC-AUC."
_ref = cross_val_predict(modelo, X_train, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
assert proba_cv is not None and np.allclose(proba_cv, _ref, atol=1e-8), "proba_cv debe venir de cross_val_predict sobre el ENTRENAMIENTO con el 'cv' dado y method='predict_proba' (columna de la clase 1)."
assert costos_cv is not None and len(costos_cv) == len(umbrales), "costos_cv debe tener un valor por cada umbral."
_costos = [costo(y_train, _ref >= t) for t in umbrales]
assert np.allclose(costos_cv, _costos), "costos_cv[i] debe ser el costo en ENTRENAMIENTO al predecir 1 si proba_cv >= umbrales[i]."
assert umbral_opt is not None and abs(costo(y_train, proba_cv >= umbral_opt) - min(_costos)) < 1e-9, "umbral_opt debe ser un umbral de costo mínimo en costos_cv."
assert umbral_opt < 0.2, "Con falsos negativos 40 veces más caros que los falsos positivos, el umbral óptimo debe ser mucho menor que 0.5."
assert costo_05 is not None and costo_05 == costo(y_test, _p >= 0.5), "costo_05 es el costo en PRUEBA con umbral 0.5."
assert costo_opt is not None and costo_opt == costo(y_test, _p >= umbral_opt), "costo_opt es el costo en PRUEBA con umbral_opt."
assert costo_opt < costo_05, "El umbral por costo debería reducir el costo en prueba."
print(f"✅ Reto 2 superado: umbral óptimo = {umbral_opt:.2f} | costo en prueba {costo_05:,.0f} → {costo_opt:,.0f} | ROC-AUC = {auc:.3f}, PR-AUC = {ap:.3f}")

---
## Reto 3: Métricas de regresión a mano y el valor de una línea base 📏

**Contexto:** la empresa de acueducto compara tres modelos del consumo diario de agua (m³). Ya están entrenados: una línea base ingenua (`DummyRegressor`, predice la media), un `Ridge` lineal y un bosque. Los datos incluyen un 2 % de lecturas atípicas (fugas o errores).

**Enunciado**
1. Implementa `metricas(y_real, y_pred)` con NumPy que devuelva un diccionario con las claves `"mse"`, `"rmse"`, `"mae"`, `"mape"` y `"r2"`:
   $$\mathrm{MSE}=\tfrac1n\sum (y_i-\hat y_i)^2,\quad \mathrm{MAE}=\tfrac1n\sum |y_i-\hat y_i|,\quad \mathrm{MAPE}=\tfrac1n\sum \frac{|y_i-\hat y_i|}{|y_i|},\quad R^2 = 1-\frac{\sum (y_i-\hat y_i)^2}{\sum (y_i-\bar y)^2}.$$
2. Calcula `m_dummy`, `m_ridge` y `m_rf` sobre el conjunto de **prueba** (`yr_test`).
3. `mejor_modelo_rmse`: `"Dummy"`, `"Ridge"` o `"Bosque"`, según el menor RMSE.

**Criterios de aceptación:** tus métricas coinciden con las de `scikit-learn`; el $R^2$ de la línea base es casi 0; el RMSE de `Ridge` es menos de la mitad que el de la línea base.

**Puntaje sugerido:** 20 puntos.

In [ ]:
# Modelos ya entrenados (dados)
dummy = DummyRegressor().fit(Xr_train, yr_train)
ridge = Ridge().fit(Xr_train, yr_train)
rf = RandomForestRegressor(n_estimators=100, random_state=0).fit(Xr_train, yr_train)


def metricas(y_real, y_pred):
    # TODO: calcula con NumPy y devuelve {"mse": ..., "rmse": ..., "mae": ..., "mape": ..., "r2": ...}
    return None


# TODO: completa
m_dummy = None            # metricas(yr_test, predicciones de dummy en prueba)
m_ridge = None
m_rf = None
mejor_modelo_rmse = None  # "Dummy" | "Ridge" | "Bosque"

In [ ]:
# Autoverificación del Reto 3 (no modificar)
def _sk(y, p):
    return {"mse": mean_squared_error(y, p), "rmse": mean_squared_error(y, p) ** 0.5, "mae": mean_absolute_error(y, p),
            "mape": mean_absolute_percentage_error(y, p), "r2": r2_score(y, p)}

_toy = metricas(np.array([100.0, 200.0, 300.0, 400.0]), np.array([110.0, 180.0, 330.0, 380.0]))
assert isinstance(_toy, dict) and set(_toy) == {"mse", "rmse", "mae", "mape", "r2"}, "metricas debe devolver un dict con las claves 'mse', 'rmse', 'mae', 'mape' y 'r2'."
assert all(abs(_toy[k] - v) < 1e-9 for k, v in _sk(np.array([100.0, 200.0, 300.0, 400.0]), np.array([110.0, 180.0, 330.0, 380.0])).items()), "metricas no coincide con scikit-learn en un ejemplo pequeño: revisa las fórmulas (MAPE es una fracción, no un porcentaje; R² usa la media de y_real)."
for _nombre, _m, _modelo in [("Dummy", m_dummy, dummy), ("Ridge", m_ridge, ridge), ("Bosque", m_rf, rf)]:
    assert isinstance(_m, dict), f"Calcula m_{_nombre.lower()} con metricas(yr_test, predicciones)."
    assert all(abs(_m[k] - v) < 1e-8 for k, v in _sk(yr_test, _modelo.predict(Xr_test)).items()), f"Las métricas de {_nombre} no coinciden con las de scikit-learn sobre el conjunto de prueba."
assert abs(m_dummy["r2"]) < 0.05, "La línea base (predecir la media) debe tener R² ≈ 0."
assert m_ridge["rmse"] < 0.5 * m_dummy["rmse"], "Ridge debería reducir el RMSE de la línea base a menos de la mitad."
_rm = {"Dummy": m_dummy["rmse"], "Ridge": m_ridge["rmse"], "Bosque": m_rf["rmse"]}
assert mejor_modelo_rmse == min(_rm, key=_rm.get), "mejor_modelo_rmse debe ser el nombre del modelo con menor RMSE."
print(pd.DataFrame({"Dummy": m_dummy, "Ridge": m_ridge, "Bosque": m_rf}).round(3))
print(f"✅ Reto 3 superado: el mejor por RMSE es «{mejor_modelo_rmse}».")

---
## Reto 4: GridSearch sin fuga de datos 🔎

**Contexto:** quieres ajustar la regularización `C` y el peso de clases de la regresión logística del detector de fraude. Con clases raras, la métrica de ajuste debe ser la **PR-AUC** (`"average_precision"`), y todo el preprocesamiento debe vivir **dentro del `Pipeline`** para que la búsqueda no filtre información de las particiones de validación.

**Enunciado**
1. `pipe_gs`: `Pipeline` con `StandardScaler` (paso `"esc"`) y `LogisticRegression(max_iter=1000)` (paso `"lg"`).
2. `param_grid`: `C` en `[0.01, 0.1, 1, 10, 100]` y `class_weight` en `[None, "balanced"]` (usa los nombres `lg__C` y `lg__class_weight`).
3. `grid = GridSearchCV(pipe_gs, param_grid, cv=cv, scoring="average_precision")` ajustado con `X_train, y_train` (el `cv` estratificado ya está definido).
4. `n_candidatos` (combinaciones evaluadas), `mejor_cv` (mejor puntaje de validación cruzada) y `ap_test` (PR-AUC del mejor modelo en **prueba**, una sola vez).

**Criterios de aceptación:** 10 candidatos × 5 particiones; el mejor puntaje de validación supera 0.25 (el azar daría ≈ la proporción de fraudes, 0.05); y validación y prueba difieren menos de 0.15.

**Puntaje sugerido:** 25 puntos.

In [ ]:
# TODO: completa
pipe_gs = None      # Pipeline([("esc", StandardScaler()), ("lg", LogisticRegression(max_iter=1000))])
param_grid = None   # {"lg__C": [...], "lg__class_weight": [...]}
grid = None         # GridSearchCV(..., cv=cv, scoring="average_precision") ajustado con X_train, y_train

n_candidatos = None # número de combinaciones evaluadas
mejor_cv = None     # mejor puntaje medio de validación cruzada
ap_test = None      # PR-AUC del mejor modelo en prueba

In [ ]:
# Autoverificación del Reto 4 (no modificar)
assert isinstance(pipe_gs, Pipeline) and [n for n, _ in pipe_gs.steps] == ["esc", "lg"], "'pipe_gs' debe ser un Pipeline con los pasos 'esc' (StandardScaler) y 'lg' (LogisticRegression), en ese orden."
assert isinstance(param_grid, dict) and set(param_grid) == {"lg__C", "lg__class_weight"}, "'param_grid' debe tener exactamente las claves 'lg__C' y 'lg__class_weight'."
assert sorted(param_grid["lg__C"]) == [0.01, 0.1, 1, 10, 100] and set(map(str, param_grid["lg__class_weight"])) == {"None", "balanced"}, "Revisa los valores de C y class_weight pedidos."
assert isinstance(grid, GridSearchCV) and hasattr(grid, "best_estimator_"), "'grid' debe ser un GridSearchCV ya ajustado (con .fit(X_train, y_train))."
assert grid.scoring == "average_precision" and grid.n_splits_ == 5, "El GridSearchCV debe usar scoring='average_precision' y el 'cv' de 5 particiones dado."
assert isinstance(grid.estimator, Pipeline), "La búsqueda debe hacerse sobre el Pipeline completo (para no filtrar información entre particiones)."
assert n_candidatos == 10 == len(grid.cv_results_["params"]), "Debe haber 5 valores de C × 2 de class_weight = 10 candidatos."
assert abs(mejor_cv - grid.best_score_) < 1e-12 and mejor_cv > 0.25, "mejor_cv debe ser grid.best_score_ y superar 0.25."
assert abs(ap_test - average_precision_score(y_test, grid.predict_proba(X_test)[:, 1])) < 1e-9, "ap_test debe ser la PR-AUC de grid (el mejor modelo) en el conjunto de prueba."
assert abs(mejor_cv - ap_test) < 0.15, "La validación cruzada y la prueba deberían diferir en menos de 0.15: si no, revisa que no hayas usado la prueba para ajustar."
print(f"✅ Reto 4 superado: mejores parámetros = {grid.best_params_} | PR-AUC validación = {mejor_cv:.3f} | prueba = {ap_test:.3f}")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Por qué una exactitud tan alta no basta en el detector de fraude? Usa tus cifras de `recall`, `precision` y `acc_trivial`.
- ¿Cómo cambió el costo al mover el umbral y por qué fue legítimo elegirlo con validación cruzada del entrenamiento y no con la prueba?
- En la regresión, ¿por qué el RMSE resulta mayor que el MAE y qué papel juegan los valores atípicos? ¿Qué te dice el $R^2$ de la línea base?
- ¿Qué diferencia hay entre `mejor_cv` y `ap_test`, y qué podrías concluir sobre el modelo que desplegarías?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: matriz de confusión y métricas calculadas a mano, paradoja de la exactitud | 20 |
| Reto 2: costo, umbral elegido por validación cruzada, ROC-AUC vs. PR-AUC | 20 |
| Reto 3: métricas de regresión a mano y comparación contra la línea base | 20 |
| Reto 4: `GridSearchCV` sobre `Pipeline`, `scoring` adecuado y reporte validación vs. prueba | 25 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] El umbral y los hiperparámetros se eligieron con validación cruzada del entrenamiento; la prueba se usó solo para reportar.
- [ ] Todo el preprocesamiento (escalado) vive dentro del `Pipeline`.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Machine Learning</i>
  </p>
</div>